# Empirical Assignment 1

## Building the dataset:


In [1]:
import pandas as pd
import numpy as np
import pandas_datareader.data as web
from pandas_datareader.famafrench import get_available_datasets
import matplotlib.pyplot as plt
import pickle
import statsmodels.api as sm

In [ ]:
"""
The following FF datasets will give you the data you need to calculate the performance of the strategies
specified in the homework.
"""
ff_ds_list = ['F-F_Research_Data_5_Factors_2x3',
              'F-F_Momentum_Factor',
              'F-F_ST_Reversal_Factor',
              'Portfolios_Formed_on_AC',
              'Portfolios_Formed_on_NI',
              'Portfolios_Formed_on_VAR',
              'Portfolios_Formed_on_RESVAR']

### Extract the monthly **Excess-Market, SMB, HML** returns and **one-month T-bill rate** from dataset.

In [56]:
# get a list all of the available datasets
ds_list = get_available_datasets()

ds =  'F-F_Research_Data_Factors'
tmp = web.DataReader(ds,'famafrench',start='1920-01-01')
print(tmp['DESCR'])
ff3 = 0.01*tmp[0]  # dataframe of monthly 3 factor returns from 1926.7 to 2022.12, index = Date 
print(ff3.describe())
rf = ff3.pop('RF') 

F-F Research Data Factors
-------------------------

This file was created by CMPT_ME_BEME_RETS using the 202212 CRSP database. The 1-month TBill return is from Ibbotson and Associates, Inc. Copyright 2022 Kenneth R. French

  0 : (1158 rows x 4 cols)
  1 : Annual Factors: January-December (96 rows x 4 cols)
            Mkt-RF          SMB          HML           RF
count  1158.000000  1158.000000  1158.000000  1158.000000
mean      0.006677     0.001893     0.003608     0.002663
std       0.053534     0.031693     0.035625     0.002516
min      -0.291300    -0.172300    -0.139700    -0.000600
25%      -0.020150    -0.016000    -0.013875     0.000300
50%       0.010600     0.000800     0.001300     0.002200
75%       0.036500     0.017575     0.017600     0.004200
max       0.388500     0.365600     0.356100     0.013500


### Extract the monthly RMW and CMA returns from the dataset 

In [63]:
ds =   'F-F_Research_Data_5_Factors_2x3'
tmp = web.DataReader(ds,'famafrench',start='1920-01-01')
print(tmp['DESCR'])
ff5 = 0.01*tmp[0].iloc[:,3:5] 
print(ff5.describe())

F-F Research Data 5 Factors 2x3
-------------------------------

This file was created by CMPT_ME_BEME_OP_INV_RETS using the 202212 CRSP database. The 1-month TBill return is from Ibbotson and Associates Inc.

  0 : (714 rows x 6 cols)
  1 : Annual Factors: January-December (59 rows x 6 cols)
              RMW         CMA
count  714.000000  714.000000
mean     0.002817    0.003009
std      0.022231    0.020560
min     -0.187300   -0.069400
25%     -0.007875   -0.009975
50%      0.002400    0.001050
75%      0.013100    0.015200
max      0.130900    0.090500


### Extract the monthly UMD returns from the dataset labeled Mom

In [64]:
ds =    'F-F_Momentum_Factor'
tmp = web.DataReader(ds,'famafrench',start='1920-01-01')
print(tmp['DESCR'])
UMD = 0.01*tmp[0]
print(UMD.describe())

F-F Momentum Factor
-------------------

This file was created by CMPT_ME_PRIOR_RETS using the 202212 CRSP database. It contains a momentum factor, constructed from six value-weight portfolios formed using independent sorts on size and prior return of NYSE, AMEX, and NASDAQ stocks. Mom  is the average of the returns on two (big and small) high prior return portfolios minus the average of the returns on two low prior return portfolios. The portfolios are constructed monthly. Big means a firm is above the median market cap on the NYSE at the end of the previous month; small firms are below the median NYSE market cap. Prior return is measured from month -12 to - 2. Firms in the low prior return portfolio are below the 30th NYSE percentile. Those in the high portfolio are above the 70th NYSE percentile. Missing data are indicated by -99.99 or -999. Copyright 2022 Kenneth R. French

  0 : (1152 rows x 1 cols)
  1 : Annual Factors: January-December (96 rows x 1 cols)
            Mom   
count

### Extract the monthly short-term reversal factor from the dataset labeled ST rev

In [66]:
ds = 'F-F_ST_Reversal_Factor'
tmp = web.DataReader(ds,'famafrench',start='1920-01-01')
print(tmp['DESCR'])
ST_Rev = 0.01*tmp[0]
print(ST_Rev.describe())

F-F ST Reversal Factor
----------------------

This file was created by CMPT_ME_PRIOR_RETS using the 202212 CRSP database. It contains a momentum factor, constructed from six value-weight portfolios formed using independent sorts on size and prior return of NYSE, AMEX, and NASDAQ stocks. ST_Rev is the average of the returns on two (big and small) low prior return portfolios minus the average of the returns on two high prior return portfolios. The portfolios are constructed monthly. Big means a firm is above the median market cap on the NYSE at the end of the previous month; small firms are below the median NYSE market cap. Prior return is measured from month - 1 to - 1. Firms in the low prior return portfolio are below the 30th NYSE percentile. Those in the high portfolio are above the 70th NYSE percentile. Missing data are indicated by -99.99 or -999. Copyright 2022 Kenneth R. French

  0 : (1163 rows x 1 cols)
  1 : Annual Factors: January-December (96 rows x 1 cols)
            ST_R

### Get the updated BAB monthly returns from AQR library

In [5]:
"""
Add in the AQR BAB data
"""
aqr1 = 'E:\\behavioral finance\\hw1\\Quality Minus Junk Factors Monthly.xlsx'
aqr2 = 'E:\\behavioral finance\\hw1\\Betting Against Beta Equity Factors Monthly.xlsx'
bab_raw = pd.read_excel(aqr2,sheet_name='BAB Factors',header=18,index_col=0,parse_dates=True)
bab_raw.index = bab_raw.index.to_period('M')
bab = bab_raw['USA']; bab.name='BAB'

In [67]:
bab

DATE
1930-12   -0.000558
1931-01   -0.022446
1931-02   -0.077423
1931-03    0.029235
1931-04   -0.012986
             ...   
2022-07   -0.024981
2022-08   -0.018163
2022-09   -0.010658
2022-10    0.021148
2022-11    0.006886
Freq: M, Name: BAB, Length: 1104, dtype: float64

### Get QMJ factor from AQR library

In [68]:
qmj_raw = pd.read_excel(aqr1,sheet_name='QMJ Factors',header=18,index_col=0,parse_dates=True)
qmj_raw.index = qmj_raw.index.to_period('M')
QMJ = qmj_raw['USA']; QMJ.name='QMJ'

In [69]:
QMJ

DATE
1957-07    0.011154
1957-08    0.004877
1957-09    0.007009
1957-10    0.002708
1957-11   -0.008971
             ...   
2022-07   -0.034421
2022-08   -0.038553
2022-09    0.039832
2022-10    0.036213
2022-11    0.045862
Freq: M, Name: QMJ, Length: 785, dtype: float64

### Get returns for portfolios sorted on 1. net insurance 2.accruals 3. total volatility

In [97]:
ff_list = ['Portfolios_Formed_on_NI',  
              'Portfolios_Formed_on_AC',
              'Portfolios_Formed_on_VAR',
              'Portfolios_Formed_on_RESVAR']
NI = 0; AC = 0; TV =0; IV = 0;
name=[NI,AC,TV,IV]
name_ = ['NI','AC','TV','IV']
for i in range(4):
    ds = ff_list[i]
    tmp = web.DataReader(ds,'famafrench',start='1920-01-01')
    print(tmp['DESCR'])
    name[i] = 0.01*(tmp[0].loc[:,'Lo 20'] -tmp[0].loc[:,'Hi 20'])
    name[i].name = name_[i]
    print(name[i].describe())
NI = name[0]
AC= name[1]
TV= name[2]
IV= name[3]

Portfolios Formed on NI
-----------------------

This file was created by CMPT_EP_CFP_OP_INV_NI_AC_RETS using the 202212 CRSP database. It contains value- and equal-weighted returns for portfolios formed on NI. The portfolios are constructed at the end of June. NI (net stock issues), the change in the natural log of the split-adjusted shares outstanding from the fiscal year-end in t-2 to t-1. Annual returns are from January to December. Missing data are indicated by -99.99 or -999. The break points include utilities and include financials. The portfolios include utilities and include financials. Copyright 2022 Kenneth R. French

  0 : Value Weighted Returns -- Monthly (714 rows x 17 cols)
  1 : Equal Weighted Returns -- Monthly (714 rows x 17 cols)
  2 : Value Weighted Returns -- Annual from January to December (59 rows x 17 cols)
  3 : Equal Weighted Returns -- Annual from January to December (59 rows x 17 cols)
  4 : Number of Firms in Portfolios (714 rows x 17 cols)
  5 : Average Fi

In [89]:
TV

Date
1963-07    0.0204
1963-08   -0.0325
1963-09    0.0066
1963-10   -0.0046
1963-11    0.0248
            ...  
2022-08   -0.0213
2022-09    0.0498
2022-10    0.0582
2022-11    0.0275
2022-12    0.0524
Freq: M, Length: 714, dtype: float64

## Calculations:

### Calculate mean, volatility, and sharp ratio of each factor

In [98]:
"""
join the FF and the BAB data
"""
ptr = ff3.join(ff5).join(UMD).join(ST_Rev).join(bab).join(QMJ).join(NI).join(AC).join(TV).join(IV)

print('\nThe annualized Means of the portfolios are:\n')
print(12*ptr.mean())

print('\nThe annualized Volatilities of the portfolios are:\n')
print(np.sqrt(12)*ptr.std())

print('\nThe annualized Sharpe Ratios of the portfolios are:\n')
print(np.sqrt(12)*(ptr.mean()-rf)/ptr.std())


The annualized Means of the portfolios are:

Mkt-RF    0.080130
SMB       0.022714
HML       0.043294
RMW       0.033798
CMA       0.036113
Mom       0.077733
ST_Rev    0.082178
BAB       0.081725
QMJ       0.045495
NI        0.038654
AC        0.035608
TV        0.035287
IV        0.035943
dtype: float64

The annualized Volatilities of the portfolios are:

Mkt-RF    0.185449
SMB       0.109789
HML       0.123408
RMW       0.077010
CMA       0.071223
Mom       0.162541
ST_Rev    0.119165
BAB       0.112208
QMJ       0.077576
NI        0.103101
AC        0.080350
TV        0.220819
IV        0.206276
dtype: float64

The annualized Sharpe Ratios of the portfolios are:

Mkt-RF    0.432085
SMB       0.206887
HML       0.350823
RMW       0.438884
CMA       0.507035
Mom       0.478238
ST_Rev    0.689619
BAB       0.728342
QMJ       0.586452
NI        0.374911
AC        0.443167
TV        0.159802
IV        0.174247
dtype: float64


### Calculate the return correlation matrix

In [129]:
print('\nThe correlation matrix for the portfolio returns is:\n')
lmat = np.tril(ptr.corr(),k=-1)
print(ptr.corr())
print('\nThe maximum correlation is  is:\n')
max_ind = np.unravel_index(np.argmax(lmat), lmat.shape)
print(lmat[max_ind])
print(f'\n {ptr.columns[  max_ind[0]  ] } and { ptr.columns[ max_ind[1]  ]} have smallest correlation \n')

print('\nThe minimum correlation is :\n')
min_ind = np.unravel_index(np.argmin(lmat), lmat.shape)
print(lmat[min_ind])

print(f'\n {ptr.columns[  min_ind[0]  ] } and { ptr.columns[ min_ind[1]  ]} have smallest correlation \n')


The correlation matrix for the portfolio returns is:

          Mkt-RF       SMB       HML       RMW       CMA    Mom       ST_Rev  \
Mkt-RF  1.000000  0.315106  0.230983 -0.180255 -0.365745 -0.344458  0.222279   
SMB     0.315106  1.000000  0.113972 -0.406868 -0.177690 -0.151484  0.168739   
HML     0.230983  0.113972  1.000000  0.090469  0.682734 -0.415131  0.046924   
RMW    -0.180255 -0.406868  0.090469  1.000000 -0.019135  0.080389 -0.085688   
CMA    -0.365745 -0.177690  0.682734 -0.019135  1.000000 -0.021060 -0.133606   
Mom    -0.344458 -0.151484 -0.415131  0.080389 -0.021060  1.000000 -0.210158   
ST_Rev  0.222279  0.168739  0.046924 -0.085688 -0.133606 -0.210158  1.000000   
BAB    -0.132186 -0.045881 -0.058080  0.292448  0.294519  0.293689 -0.078235   
QMJ    -0.492754 -0.490697 -0.022017  0.706536  0.090753  0.272573 -0.281943   
NI     -0.372445 -0.459323  0.311689  0.484188  0.446723  0.179854 -0.217648   
AC     -0.096346 -0.218682 -0.010005 -0.066618  0.082750  0.15101

??? why ???

### Calculate mean-variance efficient portfolio weights

In [151]:
u = np.array(ptr.mean()).reshape(13,1)
r = np.ones((13,1) ) * rf.mean()
C = np.array(ptr.corr()).reshape(13,13)
w =  0.2 / np.matmul(np.matmul( w ,C ),w.T) * np.matmul(   (u-r).T  , np.linalg.inv(C) ) / ( np.matmul( np.matmul(   (u-r).T  , np.linalg.inv(C) ),u))

In [156]:
w[0,3]

-0.0017074309848191043

### Run a time series regression 

In [169]:
ptr['MVE'] = ptr['Mkt-RF'] * w[0,0] + ptr.SMB* w[0,1] + ptr.HML* w[0,2] +ptr.RMW* w[0,3] + ptr.CMA* w[0,4] + ptr['Mom   ']* w[0,5] + ptr.ST_Rev* w[0,6] + ptr.BAB* w[0,7] + ptr.QMJ* w[0,8] +ptr.NI* w[0,9] + ptr.AC* w[0,10] + ptr.TV* w[0,11] + ptr.IV* w[0,12]
"""
Regress the return of HML on the excess market return - Note that the statmodels regression package
won't work if there are any missing observations, so you need to drop any rows with missing data using dropna
"""
regdata = ptr[['MVE','HML']].dropna(how='any')
x = sm.add_constant(regdata['MVE'])
y = regdata['HML']
model = sm.OLS(y,x)
results = model.fit(cov_type='HAC',cov_kwds={'maxlags':6}) #I do Newey-West corrected std. errors, with 6 lags
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                    HML   R-squared:                       0.026
Model:                            OLS   Adj. R-squared:                  0.025
Method:                 Least Squares   F-statistic:                     8.535
Date:                Sun, 12 Feb 2023   Prob (F-statistic):            0.00359
Time:                        23:07:25   Log-Likelihood:                 1504.5
No. Observations:                 713   AIC:                            -3005.
Df Residuals:                     711   BIC:                            -2996.
Df Model:                           1                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0048      0.002      2.963      0.0

### Split full sample and calculate sharpe ratio

In [172]:
print('\nThe annualized Sharpe Ratios of the portfolios in firt half period are:\n')
print(np.sqrt(12)*ptr.iloc[:650,].mean()/ptr.std())

print('\nThe annualized Sharpe Ratios of the portfolios in last half period are:\n')
print(np.sqrt(12)*ptr.iloc[650:,].mean()/ptr.std())


The annualized Sharpe Ratios of the portfolios in firt half period are:

Mkt-RF    0.431931
SMB       0.311742
HML       0.422688
RMW       0.046067
CMA       0.478709
Mom       0.534653
ST_Rev    1.011270
BAB       0.527614
QMJ       0.235550
NI        0.259901
AC        0.678513
TV       -0.098820
IV       -0.090990
MVE       1.090153
dtype: float64

The annualized Sharpe Ratios of the portfolios in last half period are:

Mkt-RF    0.432282
SMB       0.072724
HML       0.258869
RMW       0.598176
CMA       0.518521
Mom       0.406720
ST_Rev    0.278057
BAB       0.964702
QMJ       0.778859
NI        0.421549
AC        0.347731
TV        0.264677
IV        0.281803
MVE       1.254011
dtype: float64


Sharpe Ratio decreased